In [11]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [12]:
import os
import pandas as pd
from tqdm import tqdm

# Allow more columns to be displayed
pd.set_option("display.max_columns", None)

import logging
logging.basicConfig(level=logging.WARNING)

In [13]:
# Allow imports from parent directory
import sys
sys.path.append('..')
from utils.flood_request_utils import (
    # Hazard
    get_wri_and_si_hazard_data,
    plot_wri_and_si_hazard_data,

    # Vulnerability
    plot_wri_and_si_vulnerability_data,
    apply_damage_fraction,
    get_damage_fraction
)

In [14]:
!which python

/Users/klemenkubelj/miniconda3/envs/cvar-masters/bin/python


In [15]:
DATA_DIR = "../data"

In [16]:
vloge_df = pd.read_csv(os.path.join(DATA_DIR, "vloge_processed_with_gps_filtered_2025-05-10.csv"))
vloge_df.shape

(13531, 17)

In [17]:
vloge_df.head()

,VlogaId,Objekt_Naslov,Objekt_Naslov_PostnaStevilka,Objekt_Parcela_Stevilka,Objekt_Parcela_KoId,Objekt_UporabnaPovrsina,Objekt_VisinaVodeCm,Objekt_StopnjaPoskodovanosti,Skoda_DatumOcene,DogodekId,Vrednost,OdstPoskodovanostiObjekta,SkupnaSkoda,SkupnaSkodaSource,geometry,gps_lat,gps_lng
0,147651,Cesta med vinogradi 48,6000 KOPER - CAPODISTRIA,5743/6,715,50.0,30.0,NaN,09/30/10 00:00:00,14,NaN,NaN,NaN,NaN,POINT (13.786125896465217 45.55509406325257),45.555094,13.786126
1,147686,Cesta med vinogradi 46,6000 KOPER - CAPODISTRIA,5743/10,715,55.0,50.0,NaN,09/30/10 00:00:00,14,NaN,NaN,NaN,NaN,POINT (13.785861234253835 45.55489328818974),45.554893,13.785861
2,147692,Cesta med vinogradi 46,6000 KOPER - CAPODISTRIA,5743/10,715,38.0,50.0,NaN,09/30/10 00:00:00,14,NaN,NaN,NaN,NaN,POINT (13.785861234253835 45.55489328818974),45.554893,13.785861
3,147704,Cesta med vinogradi 48,6000 KOPER - CAPODISTRIA,5743/6,715,80.0,40.0,NaN,09/30/10 00:00:00,14,NaN,NaN,NaN,NaN,POINT (13.786125896465217 45.55509406325257),45.555094,13.786126
4,147723,Železniška cesta 8,6280 ANKARAN - ANCARANO,877/7,705,290.0,130.0,NaN,10/01/10 00:00:00,14,NaN,NaN,NaN,NaN,POINT (13.756193519543181 45.562016308567),45.562016,13.756194


In [19]:
def calc_predicted_flood_depth(row):
    # Get the hazard data
    gps = {
        "lat": row["gps_lat"],
        "lng": row["gps_lng"]
    }
    data, request = get_wri_and_si_hazard_data(gps)
    return (
        data["flood_depths"]["wri"][100],
        data["flood_depths"]["si"][100],
        data["flood_depths"]["si_old"][100],
        data["damage_fractions"]["wri"][100],
        data["damage_fractions"]["si"][100],
        data["damage_fractions"]["si_old"][100]
    )

_df = vloge_df.copy() # .head(10)
# Create a tqdm progress bar for the DataFrame rows
tqdm.pandas(desc="Processing rows")
# Apply the function with progress bar
results = _df.progress_apply(calc_predicted_flood_depth, axis=1)

# Add 
(
    _df["predicted_wri_depth"],
    _df["predicted_si_depth"],
    _df["predicted_si_old_depth"],
    _df["predicted_wri_damage"],
    _df["predicted_si_damage"],
    _df["predicted_si_old_damage"]
)  = zip(*results)

Processing rows: 100%|██████████| 13531/13531 [30:35<00:00,  7.37it/s] 


## Export

In [21]:
_df["measured_depth"] = _df["Objekt_VisinaVodeCm"]/100

# Export all cols but "gps_point"
_df.to_excel(os.path.join("../data/predicted_flood_depths_2025-05-13.xlsx"), index=False)